Visualize target bezier curve for convergence test of 2D shape loss

In [82]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

import path_settings
from catheter_reconstruction.utils import *
import camera_settings

In [83]:
def image_path(exp_name, i):
    data_alias = 'D' + str(0).zfill(2)
    method_dir = os.path.join(path_settings.results_dir, exp_name)
    data_dir = os.path.join(method_dir, data_alias + '_' + str(i).zfill(4))
    images_save_dir = os.path.join(data_dir, 'images')
    path1 = os.path.join(images_save_dir, '000.png')
    path2 = os.path.join(images_save_dir, '010.png')
    
    return path1, path2

In [84]:
def param_path(exp_name, i):
    data_alias = 'D' + str(0).zfill(2)
    method_dir = os.path.join(path_settings.results_dir, exp_name)
    data_dir = os.path.join(method_dir, data_alias + '_' + str(i).zfill(4))
    path = os.path.join(data_dir, 'p3d_poses.npy')
    
    return path

In [85]:
def convert_3d_to_2d(p, camera_extrinsics, fx, fy, cx, cy):
    """
    Convert 3D points to 2D points
    """ 
    p_4d = np.append(p, 1)
    p_cam = camera_extrinsics @ p_4d
    p_x = p_cam[0] * fx / p_cam[2] + cx
    p_y = p_cam[1] * fy / p_cam[2] + cy
    p_2d = np.array([p_x, p_y])
    
    # Convert from pixel coordinate frame in Blender to OpenCV
    p_2d[0] = round(p_2d[0])
    p_2d[1] = round(480 - p_2d[1])
    
    return p_2d

In [86]:
def plot_image(image_path, p0_3d, p1_3d, p2_3d, title, show=True):
    control_points = np.vstack([p0_3d, p1_3d, p2_3d])

    bezier_3d = bezier_curve_3d(control_points)

    bezier_2d = np.zeros((bezier_3d.shape[0], 2))
    for i in range(bezier_3d.shape[0]):
        bezier_2d[i] = convert_3d_to_2d(bezier_3d[i], camera_settings.extrinsics, camera_settings.a, camera_settings.b, camera_settings.center_x, camera_settings.center_y)
        
    p2_2d = convert_3d_to_2d(p2_3d, camera_settings.extrinsics, camera_settings.a, camera_settings.b, camera_settings.center_x, camera_settings.center_y)
    p1_2d = convert_3d_to_2d(p1_3d, camera_settings.extrinsics, camera_settings.a, camera_settings.b, camera_settings.center_x, camera_settings.center_y)
    p0_2d = convert_3d_to_2d(p0_3d, camera_settings.extrinsics, camera_settings.a, camera_settings.b, camera_settings.center_x, camera_settings.center_y)
        
    bezier_points = bezier_2d[1:, :]
    # bezier_points = bezier_2d

    image = plt.imread(image_path)
    plt.imshow(image)

    plt.plot(bezier_points[:, 0], bezier_points[:, 1], color='red', linewidth=2, label='Target Bezier Curve')

    plt.scatter([p0_2d[0], p1_2d[0], p2_2d[0]], [p0_2d[1], p1_2d[1], p2_2d[1]], color='blue', label='Control Points')
    # Connect control points with dashed lines (P0 to P1, P1 to P2)
    plt.plot([p0_2d[0], p1_2d[0]], [p0_2d[1], p1_2d[1]], linestyle='--', color='blue')
    plt.plot([p1_2d[0], p2_2d[0]], [p1_2d[1], p2_2d[1]], linestyle='--', color='blue')

    # Set the axis limits to match the image size
    plt.xlim(0, 640)  # X axis limit for image width
    plt.ylim(480, 0)  # Y axis limit for image height, inverted to match image coordinates

    plt.title(title)
    plt.axis('off')  
    plt.legend()
    
    dir_name, file_name = os.path.split(image_path)
    file_base, file_ext = os.path.splitext(file_name)
    new_file_name  = f"{file_base}_show{file_ext}"
    image_save_path = os.path.join(dir_name, new_file_name)
    plt.savefig(image_save_path)
    print(f"Saved image to {image_save_path}")
    
    if show:
        plt.show()
        
    plt.close()
    
    
    

In [87]:
# exp_name = "EXP008"
# i = 0
# path1, path2 = image_path(exp_name, i)
# print(path1)

# param = np.load(param_path(exp_name, i))
# p2_3d = param[-1, -1, :]
# p1_3d = param[-1, -2, :]
# p0_3d = np.array([2e-2, 2e-3, 0.00001])

# print(p2_3d)
# print(p1_3d)

# plot_image(path1, p0_3d, p1_3d, p2_3d, 'Initial Catheter Shape')
# plot_image(path2, p0_3d, p1_3d, p2_3d, 'Final Catheter Shape')

In [88]:
exp_name = "EXP007"
for i in range(20):
    path1, path2 = image_path(exp_name, i)

    param = np.load(param_path(exp_name, i))
    p2_3d = param[-1, -1, :]
    p1_3d = param[-1, -2, :]
    p0_3d = np.array([2e-2, 2e-3, 0.00001])

    plot_image(path1, p0_3d, p1_3d, p2_3d, 'Initial Catheter Shape', show=False)
    plot_image(path2, p0_3d, p1_3d, p2_3d, 'Final Catheter Shape', show=False)

Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0000\images\000_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0000\images\010_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0001\images\000_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0001\images\010_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0002\images\000_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0002\images\010_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0003\images\000_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Catheter/diff_catheter/results\EXP007\D00_0003\images\010_show.png
Saved image to E:/OneDrive - UC San Diego/UCSD/Lab/Cathe